In [ ]:
import math
from hdfs import init
from dataFrameSize import dfSizeEstimator
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [ ]:
builder = (
    SparkSession.builder
    .appName("Lakehouse")
    .master("yarn")
    .config("spark.sql.shuffle.partitions", "8")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.2.0"
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = builder.getOrCreate()
init(spark)

In [ ]:
df = spark.read.format('delta').load('hdfs:///user/jovyan/delta_lake/silver/stage_flight')

In [ ]:
%%sql var=df
select *
from delta.`hdfs:///user/jovyan/delta_lake/silver/stage_flight`;

In [ ]:
df.limit(1).show()

In [ ]:
df.createOrReplaceTempView('flight')

In [ ]:
%%sql
select * from flight limit 3;

In [ ]:
df.show(1, vertical=True)

In [ ]:
%%sql
select distinct passenger_flight_class from flight;

In [ ]:
df1 = df.select(df.aircraft_id, df.flight_cost, df.distance, df.passenger_flight_class)\
    .groupby('aircraft_id', 'passenger_flight_class')\
    .agg(F.avg('flight_cost').alias('avg_cost'), F.max('distance'))\
    .orderBy('avg_cost', ascending=False)\
    .where(F.col('passenger_flight_class') != 'economy')

In [ ]:
df1.show(5)